# M9 — Diseño BES: Flujo Secuencial de Selección de Equipos

**SIMBES · Módulo 9**  
Wizard de diseño de sistema BES (Bombeo Electrosumergible) en 8 pasos (PASO 0–7).  
Física implementada en `Module9_BESDesign/physics/`.

## Diferencia clave con M1–M8

| Variable | M1–M8 | M9 |
|---|---|---|
| **Pwf** | Resultado del balance nodal | **Entrada** — decisión de yacimientos |
| **Q** | Punto de operación IPR∩VLP | `iprPwfToQ(Pwf, Pr, Pb, IP)` |
| **Flujo** | Análisis del estado actual | **Diseño** — ¿qué equipo necesito? |

## Mapa de pasos

```
PASO 0 → Datos del sistema    (Pr, Pb, IP, Pwf, geometría, fluido, superficie)
PASO 1 → Candidatura BES       (7 criterios, veredicto: approved/conditional/rejected)
PASO 2 → IPR inversa           (Q_resultante dado Pwf, AOF, drawdown%, zona Darcy/Vogel)
PASO 3 → Condiciones en bomba  (PIP, GVF, separador, Q_total corregido)
PASO 4 → TDH + Selección bomba (TDH, serie, etapas, BEP ratio, frecuencia óptima)
PASO 5 → Motor                 (HP, corriente, T° motor, velocidad anular, shroud)
PASO 6 → Cable                 (AWG, caída de voltaje, Arrhenius, THD/IEEE 519-2014)
PASO 7 → Chequeo mecánico      (OD string vs casing, holgura, dogleg severity)
```

**CICLOS de iteración (diseño no lineal):**
- **CICLO A**: Separador → recalcula GVF y Q_total (PASO 3)
- **CICLO B**: Frecuencia / serie diferente (PASO 4)
- **CICLO C**: Agregar/quitar shroud (PASO 5)
- **CICLO D**: Cambiar AWG (PASO 6)
- **CICLO E**: OD excede casing → volver a PASO 4 con restricción OD

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import Optional

# Constantes de conversión
M3D_PER_STB   = 0.158987      # 1 STB = 0.158987 m³
M_TO_FT       = 3.28084
IP_M3D_TO_STBD = 1 / 6.28981  # 1 m³/d/psi = 0.15899 STB/d/psi
GOR_M3_TO_SCF  = 5.6146       # 1 m³/m³ = 5.6146 scf/STB

print('Librerías cargadas. Constantes de conversión definidas.')

## PASO 0 — Datos del sistema

Inputs ingresados por el ingeniero de yacimientos + datos del pozo.

In [ ]:
# ── Datos de entrada del sistema (caso ejemplo: Pozo Andes-7)
inputs = {
    # Yacimiento
    'Pr':     3000,   # psi — presión estática del reservorio
    'Pb':     1500,   # psi — presión de burbuja
    'IP':     4.0,    # m³/d/psi — índice de productividad
    'Pwf':    1000,   # psi — DECISIÓN ESTRATÉGICA (no calculado)

    # Pozo
    'D_bomba': 2000,  # m — profundidad asiento bomba
    'D_total': 2500,  # m — profundidad total
    'WHP':     150,   # psi — wellhead pressure
    'gamma':   0.40,  # psi/ft — gradiente de fluido
    'T_fond':  120,   # °C — temperatura de fondo
    'T_sup':   25,    # °C — temperatura superficial
    'ID_cas':  7.0,   # pulg — ID casing (drift)
    'Dev':     15,    # °/30m — dogleg severity máximo

    # Fluido
    'GOR':    50,     # m³/m³
    'BSW':    30,     # %
    'visc':   5,      # cp
    'API':    30,     # °API
    'H2S':    False,
    'CO2':    False,
    'solidos': 'Bajo',

    # Superficie
    'V_sup':  4160,   # V
    'f_red':  60,     # Hz
    'VSD':    'multipulse_18',
}

print('─' * 50)
print('PASO 0 — Datos del sistema (Pozo Andes-7)')
print('─' * 50)
print(f"  Pr = {inputs['Pr']} psi   Pb = {inputs['Pb']} psi   IP = {inputs['IP']} m³/d/psi")
print(f"  Pwf = {inputs['Pwf']} psi  (DECISIÓN DE YACIMIENTOS)")
print(f"  D_bomba = {inputs['D_bomba']} m   ID_cas = {inputs['ID_cas']}\"   GOR = {inputs['GOR']} m³/m³")
print(f"  BSW = {inputs['BSW']}%   T_fond = {inputs['T_fond']}°C   VSD = {inputs['VSD']}")

## PASO 1 — Candidatura BES

7 criterios de evaluación. Resultado: `approved`, `conditional`, o `rejected`.

| Criterio | Umbral OK | Umbral WARNING | Umbral BLOCKED |
|---|---|---|---|
| Caudal | 100–10 000 m³/d | — | < 100 o > 10 000 |
| Profundidad | < 4 000 m | 4 000–5 000 m | > 5 000 m |
| Temperatura | < 150°C | 150–175°C | > 175°C |
| GOR | < 100 m³/m³ | 100–300 m³/m³ | > 300 m³/m³ |
| Sólidos | Bajo | Medio | Alto |
| H₂S | Ausente | — | Presente (requiere NACE) |
| Desviación | < 45°/30m | 45–70°/30m | > 70°/30m |

In [ ]:
def ipr_pwf_to_q(Pwf, Pr, Pb, IP_m3d):
    """IPR: Q en m³/d dado Pwf. Darcy si Pwf >= Pb, Vogel si Pwf < Pb."""
    IP_stbd = IP_m3d * IP_M3D_TO_STBD
    Qb = IP_stbd * max(Pr - Pb, 0)
    AOF = Qb + (IP_stbd * Pb) / 1.8
    if Pwf >= Pb:
        Q_stbd = IP_stbd * (Pr - Pwf)
    else:
        ratio = Pwf / Pb
        Q_stbd = AOF * (1 - 0.2 * ratio - 0.8 * ratio**2)
    return max(Q_stbd, 0) * M3D_PER_STB  # m³/d

def evaluate_candidacy(inputs, Q_m3d):
    criterios = []

    # 1. Caudal
    if Q_m3d < 80:
        criterios.append({'nombre': 'Caudal', 'valor': f'{Q_m3d:.0f} m³/d', 'status': 'blocked', 'msg': 'Q < 80 m³/d → BES sobredimensionado. Evaluar plunger lift o gas lift.'})
    elif Q_m3d > 10000:
        criterios.append({'nombre': 'Caudal', 'valor': f'{Q_m3d:.0f} m³/d', 'status': 'blocked', 'msg': 'Q > 10 000 m³/d → fuera de rango BES convencional.'})
    else:
        criterios.append({'nombre': 'Caudal', 'valor': f'{Q_m3d:.0f} m³/d', 'status': 'ok', 'msg': 'Rango BES óptimo.'})

    # 2. Profundidad
    D = inputs['D_bomba']
    if D > 5000:
        criterios.append({'nombre': 'Profundidad', 'valor': f'{D} m', 'status': 'blocked', 'msg': 'D > 5000 m: limitaciones mecánicas y eléctricas severas.'})
    elif D > 4000:
        criterios.append({'nombre': 'Profundidad', 'valor': f'{D} m', 'status': 'warning', 'msg': 'D 4000–5000 m: evaluar cable flat y motor especial.'})
    else:
        criterios.append({'nombre': 'Profundidad', 'valor': f'{D} m', 'status': 'ok', 'msg': 'Profundidad dentro de rango estándar.'})

    # 3. Temperatura
    T = inputs['T_fond']
    if T > 175:
        criterios.append({'nombre': 'Temperatura', 'valor': f'{T}°C', 'status': 'blocked', 'msg': 'T > 175°C: aislamiento estándar no apto. Requiere PEEK + motor especial.'})
    elif T > 150:
        criterios.append({'nombre': 'Temperatura', 'valor': f'{T}°C', 'status': 'warning', 'msg': 'T 150–175°C: verificar rating de motor y cable EPDM/PEEK.'})
    else:
        criterios.append({'nombre': 'Temperatura', 'valor': f'{T}°C', 'status': 'ok', 'msg': 'Temperatura dentro de rating estándar.'})

    # 4. GOR
    GOR = inputs['GOR']
    if GOR > 300:
        criterios.append({'nombre': 'GOR', 'valor': f'{GOR} m³/m³', 'status': 'blocked', 'msg': 'GOR alto: gas lock probable. Requiere gas handler o AGS.'})
    elif GOR > 100:
        criterios.append({'nombre': 'GOR', 'valor': f'{GOR} m³/m³', 'status': 'warning', 'msg': 'GOR moderado: considerar AGS pasivo.'})
    else:
        criterios.append({'nombre': 'GOR', 'valor': f'{GOR} m³/m³', 'status': 'ok', 'msg': 'GOR aceptable sin separación.'})

    # 5. Sólidos
    sol = inputs['solidos']
    if sol == 'Alto':
        criterios.append({'nombre': 'Sólidos', 'valor': sol, 'status': 'blocked', 'msg': 'Abrasión alta: requiere bomba hardface y filtros.'})
    elif sol == 'Medio':
        criterios.append({'nombre': 'Sólidos', 'valor': sol, 'status': 'warning', 'msg': 'Abrasión moderada: evaluar materiales hardface.'})
    else:
        criterios.append({'nombre': 'Sólidos', 'valor': sol, 'status': 'ok', 'msg': 'Sin riesgo de abrasión significativa.'})

    # 6. H2S
    if inputs['H2S']:
        criterios.append({'nombre': 'H₂S', 'valor': 'Presente', 'status': 'warning',
                          'msg': 'H₂S: requerido Lead Sheath + Monel 400 (NACE MR0175).'})
    else:
        criterios.append({'nombre': 'H₂S', 'valor': 'Ausente', 'status': 'ok', 'msg': 'Sin gas amargo.'})

    # 7. Desviación
    dev = inputs['Dev']
    if dev > 70:
        criterios.append({'nombre': 'Desviación', 'valor': f'{dev}°/30m', 'status': 'blocked',
                          'msg': 'Dogleg > 70°/30m: riesgo mecánico severo en instalación BES.'})
    elif dev > 45:
        criterios.append({'nombre': 'Desviación', 'valor': f'{dev}°/30m', 'status': 'warning',
                          'msg': 'Dogleg 45–70°/30m: verificar radio de curvatura con proveedor.'})
    else:
        criterios.append({'nombre': 'Desviación', 'valor': f'{dev}°/30m', 'status': 'ok',
                          'msg': 'Desviación dentro de límite estándar BES.'})

    blocked = any(c['status'] == 'blocked' for c in criterios)
    warnings = any(c['status'] == 'warning' for c in criterios)
    verdict = 'rejected' if blocked else ('conditional' if warnings else 'approved')
    return {'criterios': criterios, 'verdict': verdict}

# Calcular
Q_m3d = ipr_pwf_to_q(inputs['Pwf'], inputs['Pr'], inputs['Pb'], inputs['IP'])
result1 = evaluate_candidacy(inputs, Q_m3d)

icons = {'ok': '✅', 'warning': '⚠️', 'blocked': '❌'}
print('─' * 60)
print('PASO 1 — Candidatura BES')
print('─' * 60)
for c in result1['criterios']:
    print(f"  {icons[c['status']]}  {c['nombre']:15s}  {c['valor']:15s}  {c['msg']}")
print()
verdict_label = {'approved': '✅ APROBADO', 'conditional': '⚠️ CONDICIONAL', 'rejected': '❌ RECHAZADO'}
print(f"  Veredicto: {verdict_label[result1['verdict']]}")

## PASO 2 — IPR Inversa

**Inversión pedagógica clave:** Pwf es INPUT (decisión de yacimientos), Q es resultado.

```
Q_resultante = iprPwfToQ(Pwf, Pr, Pb, IP)
```

Esto permite al ingeniero de producción:
1. Definir Pwf objetivo (cuánto drawdown aplicar)
2. Calcular Q resultante
3. Diseñar el equipo BES para entregar exactamente ese Q

In [ ]:
def calc_aof(Pr, Pb, IP_m3d):
    IP_stbd = IP_m3d * IP_M3D_TO_STBD
    Qb = IP_stbd * max(Pr - Pb, 0)
    return (Qb + (IP_stbd * Pb) / 1.8) * M3D_PER_STB

def ipr_curve(Pr, Pb, IP_m3d, n=200):
    Pwf_vals = np.linspace(0, Pr, n)
    Q_vals = np.array([ipr_pwf_to_q(p, Pr, Pb, IP_m3d) for p in Pwf_vals])
    return Pwf_vals, Q_vals

# Calcular
Q_resultante = ipr_pwf_to_q(inputs['Pwf'], inputs['Pr'], inputs['Pb'], inputs['IP'])
AOF = calc_aof(inputs['Pr'], inputs['Pb'], inputs['IP'])
drawdown_pct = (inputs['Pr'] - inputs['Pwf']) / inputs['Pr'] * 100
zona = 'Darcy' if inputs['Pwf'] >= inputs['Pb'] else 'Vogel'

Pwf_curve, Q_curve = ipr_curve(inputs['Pr'], inputs['Pb'], inputs['IP'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#0D1424')
fig.patch.set_facecolor('#0B0F1A')

ax.plot(Q_curve, Pwf_curve, color='#38BDF8', lw=2.5, label='Curva IPR')
ax.axhline(inputs['Pwf'], color='#34D399', lw=1.5, ls='--', label=f"Pwf = {inputs['Pwf']} psi (INPUT)")
ax.axhline(inputs['Pb'],  color='#FBBF24', lw=1,   ls=':', alpha=0.7, label=f"Pb = {inputs['Pb']} psi")
ax.axvline(Q_resultante,  color='#FB7185', lw=2,   ls='-', label=f"Q = {Q_resultante:.1f} m³/d")
ax.scatter([Q_resultante], [inputs['Pwf']], color='#FB7185', s=100, zorder=5)

ax.set_xlabel('Caudal Q (m³/d)', color='#CBD5E1')
ax.set_ylabel('Pwf (psi)', color='#CBD5E1')
ax.set_title('PASO 2 — IPR Inversa: Pwf como INPUT estratégico', color='#CBD5E1', fontsize=12)
ax.tick_params(colors='#64748B')
for spine in ax.spines.values(): spine.set_color('#1E293B')
ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=9)
ax.grid(color='#1E293B', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"  Q resultante  : {Q_resultante:.1f} m³/d")
print(f"  AOF           : {AOF:.1f} m³/d")
print(f"  Drawdown      : {drawdown_pct:.1f}%")
print(f"  Zona          : {zona}")

## PASO 3 — Condiciones en la Bomba

### PIP — Pump Intake Pressure

```
PIP = Pwf - γ × (D_total - D_bomba)
```

PIP es la presión en la succión de la bomba. Determina si hay gas libre.

### GVF — Gas Volume Fraction en succión

```
GVF = Q_gas_libre / (Q_liq + Q_gas_libre)
```

Calculado con factor volumétrico Bo (correlación Standing simplificada).

### Umbrales GVF
- < 10%: OK sin separación
- 10–15%: WARNING — evaluar AGS
- > 15%: DANGER — gas lock probable

In [ ]:
def calc_pip(Pwf_psi, gamma_psi_ft, D_total_m, D_bomba_m):
    """PIP en psi. gamma en psi/ft, profundidades en m."""
    delta_m = D_total_m - D_bomba_m
    delta_ft = delta_m * M_TO_FT
    PIP = Pwf_psi - gamma_psi_ft * delta_ft
    return max(PIP, 0)

def bo_standing(Pr_psi, T_C, GOR_m3m3, API):
    """Factor volumétrico Bo — correlación Standing simplificada. [SIMPLIFIED]"""
    Rs_scf = GOR_m3m3 * GOR_M3_TO_SCF
    gamma_g = 0.65
    gamma_o = 141.5 / (API + 131.5)
    T_F = T_C * 9/5 + 32
    F = Rs_scf * (gamma_g / gamma_o)**0.5 + 1.25 * T_F
    Bo = 0.972 + 0.000147 * F**1.175
    return max(Bo, 1.0)

def calc_pump_volume(Q_m3d, BSW_pct, GOR_m3m3, PIP_psi, Pb_psi, T_fond_C, API):
    """Volumen real manejado por la bomba en m³/d."""
    BSW = BSW_pct / 100
    Q_oil = Q_m3d * (1 - BSW)
    Q_water = Q_m3d * BSW

    Bo = bo_standing(PIP_psi, T_fond_C, GOR_m3m3, API)
    Bw = 1.02  # [SIMPLIFIED]

    Q_oil_res = Q_oil * Bo
    Q_water_res = Q_water * Bw
    Q_liq_res = Q_oil_res + Q_water_res

    if PIP_psi < Pb_psi:
        GOR_scf = GOR_m3m3 * GOR_M3_TO_SCF
        Rs_scf = GOR_scf * (PIP_psi / Pb_psi)
        Q_gas_scf_d = Q_oil * (GOR_scf - Rs_scf) * (1 / M3D_PER_STB)
        Bg = 0.0283 * 1.0 * (T_fond_C + 273.15) / (PIP_psi * 0.0689476)  # [SIMPLIFIED]
        Q_gas_res = max(Q_gas_scf_d * Bg, 0)
    else:
        Q_gas_res = 0.0

    Q_total = Q_liq_res + Q_gas_res
    GVF = Q_gas_res / Q_total if Q_total > 0 else 0

    return {
        'Q_total_m3d': Q_total, 'Q_liq_m3d': Q_liq_res,
        'Q_gas_m3d': Q_gas_res, 'GVF': GVF, 'Bo': Bo
    }

PIP = calc_pip(inputs['Pwf'], inputs['gamma'], inputs['D_total'], inputs['D_bomba'])
pv = calc_pump_volume(Q_resultante, inputs['BSW'], inputs['GOR'], PIP, inputs['Pb'], inputs['T_fond'], inputs['API'])

# Decisión separador
GVF = pv['GVF']
if GVF > 0.15:
    sep = 'AGS Rotary (65% eficiencia)'  # [SIMPLIFIED]
    GVF_sep = GVF * (1 - 0.65)
elif GVF > 0.10:
    sep = 'AGS Pasivo recomendado'
    GVF_sep = GVF * 0.85
else:
    sep = 'Sin separación'
    GVF_sep = GVF

print('─' * 55)
print('PASO 3 — Condiciones en la bomba')
print('─' * 55)
print(f"  PIP            : {PIP:.0f} psi")
print(f"  Q_liq (res)    : {pv['Q_liq_m3d']:.1f} m³/d")
print(f"  Q_gas (res)    : {pv['Q_gas_m3d']:.1f} m³/d")
print(f"  Q_total (res)  : {pv['Q_total_m3d']:.1f} m³/d")
print(f"  GVF en succión : {GVF*100:.1f}%  → {sep}")
print(f"  GVF post-sep   : {GVF_sep*100:.1f}%")
print(f"  Bo             : {pv['Bo']:.3f}")

## PASO 4 — TDH y Selección de Bomba

### TDH — Total Dynamic Head

```
TDH = H_static + H_friccion + H_contrapresion

H_static    = D_bomba - PIP/gamma           (columna de fluido a levantar)
H_friccion  ≈ 1.4e-5 × Q²                  (Darcy-Weisbach simplificado)
H_contrapres = WHP / gamma                  (presión en wellhead)
```

### Frecuencia óptima (Leyes de Afinidad)

```
f_opt = f_red × Q_total / Q_bep_60Hz
```

Esta fórmula no es circular — usa Q_total calculado en PASO 3.

In [ ]:
# Catálogo de series simplificado
PUMP_SERIES = [
    {'id': 'low_rate',    'OD_in': 3.75, 'BEP_stbd': 900,  'H_stage_bep_ft': 28.0},
    {'id': 'medium_rate', 'OD_in': 4.00, 'BEP_stbd': 2100, 'H_stage_bep_ft': 32.0},
    {'id': 'high_rate',   'OD_in': 5.50, 'BEP_stbd': 4500, 'H_stage_bep_ft': 35.0},
]

def calc_tdh(Q_total_m3d, PIP_psi, WHP_psi, D_bomba_m, gamma_psi_ft, f_red=60):
    """TDH en m. [SIMPLIFIED: fricción proporcional a Q²]"""
    PIP_ft = PIP_psi / gamma_psi_ft
    H_static_ft = D_bomba_m * M_TO_FT - PIP_ft
    Q_stbd = Q_total_m3d / M3D_PER_STB
    H_fric_ft  = 1.4e-5 * Q_stbd**2
    H_contra_ft = WHP_psi / gamma_psi_ft
    TDH_ft = H_static_ft + H_fric_ft + H_contra_ft
    return max(TDH_ft, 0) * 0.3048  # a metros

def select_pump(Q_total_m3d, TDH_m, ID_cas_in, f_red, OD_max=None):
    """Selecciona serie, calcula etapas y frecuencia óptima."""
    Q_stbd = Q_total_m3d / M3D_PER_STB
    results = []
    for s in PUMP_SERIES:
        if OD_max and s['OD_in'] > OD_max:
            continue
        if s['OD_in'] >= ID_cas_in:
            continue
        # Frecuencia óptima
        f_opt = np.clip(f_red * Q_stbd / s['BEP_stbd'], 40, 72)
        # Etapas
        H_stage_m = s['H_stage_bep_ft'] * (f_opt/60)**2 * 0.3048
        etapas = int(np.ceil(TDH_m / H_stage_m)) if H_stage_m > 0 else 999
        # BEP ratio
        Q_bep_op = s['BEP_stbd'] * (f_opt / 60)
        bep_pct = Q_stbd / Q_bep_op * 100 if Q_bep_op > 0 else 0
        results.append({
            'serie': s['id'], 'OD_in': s['OD_in'],
            'f_opt_Hz': round(f_opt, 1), 'etapas': etapas,
            'bep_pct': round(bep_pct, 1),
            'TDH_disp_m': round(H_stage_m * etapas, 1),
        })
    results.sort(key=lambda r: abs(r['bep_pct'] - 100))
    return results

Q_total = pv['Q_total_m3d']
TDH_m = calc_tdh(Q_total, PIP, inputs['WHP'], inputs['D_bomba'], inputs['gamma'])
candidatos = select_pump(Q_total, TDH_m, inputs['ID_cas'], inputs['f_red'])

print('─' * 65)
print('PASO 4 — TDH y Selección de Bomba')
print('─' * 65)
print(f"  TDH requerido  : {TDH_m:.0f} m  ({TDH_m * M_TO_FT:.0f} ft)")
print(f"  Q bomba        : {Q_total:.1f} m³/d  ({Q_total/M3D_PER_STB:.0f} STB/d)")
print()
print(f"  {'Serie':15s}  {'OD (in)':8s}  {'f (Hz)':7s}  {'Etapas':7s}  {'BEP %':7s}")
print(f"  {'─'*15}  {'─'*8}  {'─'*7}  {'─'*7}  {'─'*7}")
for c in candidatos:
    flag = ' ← SELECCIONADO' if c == candidatos[0] else ''
    print(f"  {c['serie']:15s}  {c['OD_in']:8.2f}  {c['f_opt_Hz']:7.1f}  {c['etapas']:7d}  {c['bep_pct']:7.1f}{flag}")

bomba_sel = candidatos[0]
print(f"\n  BEP ratio óptimo: 80–120%. Seleccionada: {bomba_sel['bep_pct']}%")

## PASO 5 — Motor

### HP Hidráulico requerido

```
HP_hid = Q_total × TDH × rho × g / eta_bomba
HP_diseño = HP_hid × 1.20   (20% margen de diseño)
```

### Velocidad anular del fluido

```
v_anular = Q_total / A_anular
A_anular = π/4 × (ID_cas² - OD_motor²)
```

La velocidad anular debe ser ≥ 0.9 m/s para refrigerar el motor. Si es menor, se requiere shroud.

In [ ]:
# Tabla de motores (5 tiers)
MOTOR_TIERS = [
    {'HP_max': 100,  'OD_in': 3.75, 'T_rated_C': 150, 'eta': 0.87, 'FP': 0.82},
    {'HP_max': 300,  'OD_in': 4.50, 'T_rated_C': 160, 'eta': 0.88, 'FP': 0.84},
    {'HP_max': 500,  'OD_in': 5.40, 'T_rated_C': 175, 'eta': 0.89, 'FP': 0.85},
    {'HP_max': 1000, 'OD_in': 6.00, 'T_rated_C': 175, 'eta': 0.90, 'FP': 0.86},
    {'HP_max': 2500, 'OD_in': 7.38, 'T_rated_C': 175, 'eta': 0.91, 'FP': 0.87},
]

def calc_motor(Q_total_m3d, TDH_m, rho_kgm3, ID_cas_in, T_fond_C, V_sup):
    eta_pump = 0.60  # [SIMPLIFIED]
    g = 9.81
    Q_m3s = Q_total_m3d / 86400
    P_hid_W = rho_kgm3 * g * TDH_m * Q_m3s / eta_pump
    HP_hid = P_hid_W / 745.7
    HP_req = HP_hid * 1.20

    tier = next((t for t in MOTOR_TIERS if t['HP_max'] >= HP_req), MOTOR_TIERS[-1])
    HP_sel = tier['HP_max']
    V_motor = V_sup
    I_nom = (HP_sel * 745.7) / (np.sqrt(3) * V_motor * tier['FP'] * tier['eta'])

    OD_motor_in = tier['OD_in']
    A_cas_m2 = np.pi/4 * (ID_cas_in * 0.0254)**2
    A_motor_m2 = np.pi/4 * (OD_motor_in * 0.0254)**2
    A_anular = A_cas_m2 - A_motor_m2
    v_anular = Q_m3s / A_anular if A_anular > 0 else 0

    # T° motor estimada
    if v_anular < 0.3:
        delta_T = 30
    elif v_anular < 0.9:
        delta_T = 15
    else:
        delta_T = 5
    T_motor_op = T_fond_C + delta_T

    shroud_req = v_anular < 0.9

    return {
        'HP_hid': HP_hid, 'HP_req': HP_req, 'HP_sel': HP_sel,
        'I_nom': I_nom, 'V_motor': V_motor,
        'OD_motor_in': OD_motor_in, 'T_rated': tier['T_rated_C'],
        'v_anular': v_anular, 'T_motor_op': T_motor_op,
        'shroud_req': shroud_req,
    }

rho_mezcla = 870  # kg/m³ [SIMPLIFIED]
m5 = calc_motor(Q_total, TDH_m, rho_mezcla, inputs['ID_cas'], inputs['T_fond'], inputs['V_sup'])

print('─' * 55)
print('PASO 5 — Motor')
print('─' * 55)
print(f"  HP hidráulico   : {m5['HP_hid']:.0f} HP")
print(f"  HP requerido    : {m5['HP_req']:.0f} HP  (+20% margen)")
print(f"  HP seleccionado : {m5['HP_sel']} HP")
print(f"  Corriente nom.  : {m5['I_nom']:.1f} A")
print(f"  OD motor        : {m5['OD_motor_in']:.2f}\"")
print(f"  T° rating motor : {m5['T_rated']}°C")
print(f"  T° operación    : {m5['T_motor_op']:.0f}°C")
print(f"  v anular        : {m5['v_anular']:.3f} m/s")
print(f"  Shroud          : {'REQUERIDO ⚠️' if m5['shroud_req'] else 'No requerido ✅'}")

## PASO 6 — Cable

### Caída de voltaje (Darcy-Weisbach eléctrico)

```
R_T = R_20 × (1 + α × (T_avg − 20))    [corrección Arrhenius térmico]
V_drop = I × R_T × 3                    [3 conductores]
V_drop_pct = V_drop / V_motor × 100
```

### Regla de Arrhenius — vida útil del aislamiento

```
life_factor = 2^((T_rated - T_operating) / 10)
```

Por cada 10°C sobre el rating nominal, la vida útil se reduce a la mitad.

### IEEE 519-2014: THD < 5% en PCC

| Topología VSD | THD típico | Cumple |
|---|---|---|
| 6 pulsos estándar | 30% | ❌ |
| 12 pulsos | 17.5% | ❌ |
| 18 pulsos | 4% | ✅ |
| AFE | 2.5% | ✅ |

In [ ]:
CABLE_R_PER_1000FT = {1: 0.1239, 2: 0.1563, 4: 0.2485, 6: 0.3951, 8: 0.6282}
CABLE_AMPACITY     = {1: 130, 2: 110, 4: 85, 6: 65, 8: 50}
VSD_THD = {
    'standard_6pulse': 30.0, 'multipulse_12': 17.5,
    'multipulse_18': 4.0,    'afe': 2.5,
}

def calc_cable(I_nom, depth_m, T_fond_C, T_sup_C, V_motor, VSD_topology):
    depth_ft = depth_m * M_TO_FT
    # Seleccionar AWG por ampacidad
    awg = next((k for k, amp in CABLE_AMPACITY.items() if amp >= I_nom * 1.25), 1)
    R_base = CABLE_R_PER_1000FT[awg] * depth_ft / 1000
    T_avg = (T_fond_C + T_sup_C) / 2
    alpha = 0.00393
    R_T = R_base * (1 + alpha * (T_avg - 20))
    V_drop = I_nom * R_T * 3
    V_drop_pct = V_drop / V_motor * 100

    # Arrhenius
    T_rated = 150  # cable estándar EPDM
    T_op = T_fond_C + 5
    delta_T = T_op - T_rated
    life_factor = 2**(-delta_T/10) if delta_T > 0 else 1.0

    # THD
    THD_pct = VSD_THD.get(VSD_topology, 30.0)
    cumple_ieee = THD_pct < 5.0

    return {
        'AWG': awg, 'R_T': R_T, 'V_drop_V': V_drop,
        'V_drop_pct': V_drop_pct, 'life_factor': life_factor,
        'THD_pct': THD_pct, 'cumple_ieee519': cumple_ieee,
    }

m6 = calc_cable(m5['I_nom'], inputs['D_bomba'], inputs['T_fond'],
                inputs['T_sup'], m5['V_motor'], inputs['VSD'])

print('─' * 55)
print('PASO 6 — Cable')
print('─' * 55)
print(f"  AWG seleccionado: {m6['AWG']}")
print(f"  Caída de voltaje: {m6['V_drop_V']:.0f} V  ({m6['V_drop_pct']:.1f}%)")
print(f"    {'✅' if m6['V_drop_pct'] <= 5 else '⚠️' if m6['V_drop_pct'] <= 10 else '❌'}  {'OK' if m6['V_drop_pct'] <= 5 else 'ADVERTENCIA' if m6['V_drop_pct'] <= 10 else 'PELIGRO'}")
print(f"  Factor vida útil: {m6['life_factor']:.3f}  ({m6['life_factor']*100:.1f}% vida nominal)")
print(f"  THD VSD          : {m6['THD_pct']}%  {'✅ Cumple IEEE 519-2014' if m6['cumple_ieee519'] else '❌ No cumple IEEE 519-2014 (5%)'}") 

# Sensibilidad: AWG vs V_drop
awg_vals = [1, 2, 4, 6, 8]
vdrop_vals = []
for awg_test in awg_vals:
    if awg_test not in CABLE_R_PER_1000FT:
        continue
    R = CABLE_R_PER_1000FT[awg_test] * (inputs['D_bomba'] * M_TO_FT) / 1000
    T_avg = (inputs['T_fond'] + inputs['T_sup']) / 2
    R_t = R * (1 + 0.00393 * (T_avg - 20))
    vd = m5['I_nom'] * R_t * 3 / m5['V_motor'] * 100
    vdrop_vals.append(vd)

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor('#0D1424'); fig.patch.set_facecolor('#0B0F1A')
ax.bar([str(a) for a in awg_vals], vdrop_vals,
       color=['#22C55E' if v <= 5 else '#F59E0B' if v <= 10 else '#EF4444' for v in vdrop_vals])
ax.axhline(5,  color='#F59E0B', ls='--', lw=1, label='Umbral 5%')
ax.axhline(10, color='#EF4444', ls='--', lw=1, label='Umbral 10%')
ax.set_xlabel('AWG (mayor número = menor conductor)', color='#CBD5E1')
ax.set_ylabel('Caída de voltaje (%)', color='#CBD5E1')
ax.set_title('Caída de voltaje vs AWG — V_drop = I × R_T × 3 / V_motor', color='#CBD5E1')
ax.tick_params(colors='#64748B')
for spine in ax.spines.values(): spine.set_color('#1E293B')
ax.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1')
ax.grid(axis='y', color='#1E293B', ls='--', alpha=0.5)
plt.tight_layout(); plt.show()

## PASO 7 — Chequeo Mecánico

### OD String vs Casing

El OD máximo del conjunto (bomba + motor + shroud si aplica) debe entrar en el casing con holgura mínima:

```
holgura = ID_cas - OD_max_string
```

- Holgura < 12 mm → WARNING (dificultad de instalación)
- Holgura ≤ 0 mm → BLOCKED (CICLO E: regresar a PASO 4 con OD_max_constraint)

### Dogleg Severity

El cable BES tolera radios de curvatura limitados según AWG. [SIMPLIFIED]

In [ ]:
DOGLEG_LIMIT = {1: 14, 2: 16, 4: 20, 6: 25, 8: 30}  # °/30m máx por AWG [SIMPLIFIED]

def check_mechanical(OD_bomba_in, OD_motor_in, OD_shroud_in, OD_cable_in, ID_cas_in, Dev_deg, AWG):
    comps = [OD_bomba_in, OD_motor_in, OD_cable_in]
    if OD_shroud_in:
        comps.append(OD_shroud_in)
    OD_max = max(comps)

    holgura_in = ID_cas_in - OD_max
    holgura_mm = holgura_in * 25.4

    dogleg_limit = DOGLEG_LIMIT.get(AWG, 20)
    dogleg_ok = Dev_deg <= dogleg_limit

    if holgura_mm <= 0:
        status = 'CICLO_E'
        msg = f'OD {OD_max:.2f}" excede ID casing {ID_cas_in:.2f}". CICLO E: volver a PASO 4 con OD_max < {OD_max:.2f}"'
    elif holgura_mm < 12:
        status = 'warning'
        msg = f'Holgura {holgura_mm:.1f} mm < 12 mm. Instalación dificultosa.'
    else:
        status = 'ok'
        msg = f'Holgura {holgura_mm:.1f} mm. OK.'

    return {
        'OD_max_in': OD_max, 'holgura_in': holgura_in, 'holgura_mm': holgura_mm,
        'dogleg_limit': dogleg_limit, 'dogleg_ok': dogleg_ok,
        'status': status, 'msg': msg,
    }

OD_bomba_in  = PUMP_SERIES[[s['id'] for s in PUMP_SERIES].index(bomba_sel['serie'])]['OD_in']
OD_motor_in  = m5['OD_motor_in']
OD_cable_est = 1.0  # pulgadas estimado AWG 4 [SIMPLIFIED]
OD_shroud_in = (OD_motor_in + 0.5) if m5['shroud_req'] else None

m7 = check_mechanical(OD_bomba_in, OD_motor_in, OD_shroud_in, OD_cable_est,
                      inputs['ID_cas'], inputs['Dev'], m6['AWG'])

print('─' * 60)
print('PASO 7 — Chequeo Mecánico')
print('─' * 60)
print(f"  OD bomba        : {OD_bomba_in:.2f}\"")
print(f"  OD motor        : {OD_motor_in:.2f}\"")
if OD_shroud_in:
    print(f"  OD shroud       : {OD_shroud_in:.2f}\"")
print(f"  OD cable        : {OD_cable_est:.2f}\"")
print(f"  OD max string   : {m7['OD_max_in']:.2f}\"")
print(f"  ID casing       : {inputs['ID_cas']:.2f}\"")
print(f"  Holgura         : {m7['holgura_mm']:.1f} mm  → {'✅ OK' if m7['status']=='ok' else '⚠️ WARNING' if m7['status']=='warning' else '❌ CICLO E'}")
print(f"  Dogleg          : {inputs['Dev']}°/30m  (límite AWG {m6['AWG']}: {m7['dogleg_limit']}°/30m)  → {'✅' if m7['dogleg_ok'] else '❌'}")
print(f"  Mensaje         : {m7['msg']}")

## Resumen — Dashboard de Diseño BES

In [ ]:
fig = plt.figure(figsize=(14, 9))
fig.patch.set_facecolor('#0B0F1A')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Colores de alerta
def c_alert(val, warn, danger, reverse=False):
    if reverse:
        return '#EF4444' if val < danger else '#F59E0B' if val < warn else '#22C55E'
    return '#EF4444' if val > danger else '#F59E0B' if val > warn else '#22C55E'

def mini_bar(ax, title, value, unit, pct, warn_pct=70, danger_pct=90, reverse=False):
    ax.set_facecolor('#111827')
    col = c_alert(pct, warn_pct, danger_pct, reverse)
    ax.barh([0], [pct], color=col + '44', height=0.5)
    ax.barh([0], [min(pct, 100)], color=col, height=0.4)
    ax.set_xlim(0, 100)
    ax.set_yticks([])
    ax.set_xlabel('%', color='#64748B', fontsize=8)
    ax.set_title(f'{title}\n{value:.1f} {unit}', color='#CBD5E1', fontsize=9)
    ax.tick_params(colors='#64748B', labelsize=7)
    for spine in ax.spines.values(): spine.set_color('#1E293B')
    ax.text(min(pct, 100) + 1, 0, f'{pct:.1f}%', va='center', color=col, fontsize=8)

# Panel 1: IPR
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor('#0D1424')
Pwf_c, Q_c = ipr_curve(inputs['Pr'], inputs['Pb'], inputs['IP'])
ax0.plot(Q_c, Pwf_c, '#38BDF8', lw=1.5)
ax0.axhline(inputs['Pwf'], color='#34D399', lw=1, ls='--')
ax0.scatter([Q_resultante], [inputs['Pwf']], color='#FB7185', s=60)
ax0.set_title('IPR — Pwf INPUT', color='#CBD5E1', fontsize=9)
ax0.set_xlabel('Q (m³/d)', color='#64748B', fontsize=7)
ax0.set_ylabel('Pwf (psi)', color='#64748B', fontsize=7)
ax0.tick_params(colors='#64748B', labelsize=7)
for s in ax0.spines.values(): s.set_color('#1E293B')
ax0.grid(color='#1E293B', ls='--', alpha=0.4)

# Panel 2: Candidatura
ax1 = fig.add_subplot(gs[0, 1])
ax1.set_facecolor('#111827')
ax1.axis('off')
verdict = result1['verdict']
vc = '#22C55E' if verdict == 'approved' else '#F59E0B' if verdict == 'conditional' else '#EF4444'
ax1.text(0.5, 0.85, 'CANDIDATURA BES', ha='center', va='center', color='#CBD5E1', fontsize=9,
         transform=ax1.transAxes, fontweight='bold')
ax1.text(0.5, 0.65, verdict.upper(), ha='center', va='center', color=vc, fontsize=14,
         transform=ax1.transAxes, fontweight='bold')
for i, c in enumerate(result1['criterios']):
    ic = {'ok': '✅', 'warning': '⚠️', 'blocked': '❌'}[c['status']]
    ax1.text(0.05, 0.50 - i*0.08, f"{ic} {c['nombre']}", color='#CBD5E1', fontsize=7.5,
             transform=ax1.transAxes)

# Panel 3: V_drop AWG
ax2 = fig.add_subplot(gs[0, 2])
mini_bar(ax2, 'Caída de Voltaje', m6['V_drop_V'], 'V', m6['V_drop_pct'], 5, 10)

# Panel 4: GVF
ax3 = fig.add_subplot(gs[1, 0])
mini_bar(ax3, 'GVF en succión', GVF * 100, '%', GVF * 100, 10, 15)

# Panel 5: Holgura mecánica
ax4 = fig.add_subplot(gs[1, 1])
holgura_pct = min(m7['holgura_mm'] / 50 * 100, 100)  # ref 50 mm
mini_bar(ax4, 'Holgura Mecánica', m7['holgura_mm'], 'mm', holgura_pct,
         warn_pct=25, danger_pct=0, reverse=True)

# Panel 6: Arrhenius
ax5 = fig.add_subplot(gs[1, 2])
life_pct = m6['life_factor'] * 100
mini_bar(ax5, 'Vida útil cable (Arrhenius)', life_pct, '%', life_pct,
         warn_pct=100, danger_pct=50, reverse=True)

fig.suptitle('M9 — Dashboard de Diseño BES · Pozo Andes-7', color='#CBD5E1',
             fontsize=12, fontweight='bold', y=0.98)
plt.show()

print('\n' + '═'*55)
print('RESUMEN EJECUTIVO — DISEÑO BES')
print('═'*55)
print(f"  Q operativo      : {Q_resultante:.1f} m³/d  (Pwf = {inputs['Pwf']} psi — INPUT)")
print(f"  TDH requerido    : {TDH_m:.0f} m")
print(f"  Serie seleccionada: {bomba_sel['serie']}  |  {bomba_sel['etapas']} etapas  |  {bomba_sel['f_opt_Hz']} Hz  |  BEP {bomba_sel['bep_pct']}%")
print(f"  Motor            : {m5['HP_sel']} HP  |  {m5['I_nom']:.1f} A  |  {'Shroud' if m5['shroud_req'] else 'Sin shroud'}")
print(f"  Cable            : AWG {m6['AWG']}  |  {m6['V_drop_pct']:.1f}% caída  |  THD {m6['THD_pct']}% {'✅' if m6['cumple_ieee519'] else '❌'}")
print(f"  Holgura casing   : {m7['holgura_mm']:.1f} mm  |  Dogleg {'✅' if m7['dogleg_ok'] else '❌'}")
print('═'*55)

## Análisis de Sensibilidad — Pwf vs Q y TDH

¿Cómo cambia el diseño si el equipo de yacimientos decide cambiar Pwf?

In [ ]:
Pwf_range = np.linspace(500, 2500, 50)
Q_range, TDH_range, GVF_range = [], [], []

for Pwf_test in Pwf_range:
    Q_t = ipr_pwf_to_q(Pwf_test, inputs['Pr'], inputs['Pb'], inputs['IP'])
    PIP_t = calc_pip(Pwf_test, inputs['gamma'], inputs['D_total'], inputs['D_bomba'])
    pv_t = calc_pump_volume(Q_t, inputs['BSW'], inputs['GOR'], PIP_t, inputs['Pb'],
                            inputs['T_fond'], inputs['API'])
    Qt = pv_t['Q_total_m3d']
    TDH_t = calc_tdh(Qt, PIP_t, inputs['WHP'], inputs['D_bomba'], inputs['gamma'])
    Q_range.append(Qt)
    TDH_range.append(TDH_t)
    GVF_range.append(pv_t['GVF'] * 100)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#0B0F1A')

for ax, yvals, ylabel, color, title in [
    (ax1, Q_range,   'Q bomba (m³/d)',    '#38BDF8', 'Q total en bomba'),
    (ax2, TDH_range, 'TDH (m)',           '#34D399', 'TDH requerido'),
    (ax3, GVF_range, 'GVF en succión (%)', '#FBBF24', 'GVF en succión'),
]:
    ax.set_facecolor('#0D1424')
    ax.plot(Pwf_range, yvals, color=color, lw=2)
    ax.axvline(inputs['Pwf'], color='#818CF8', ls='--', lw=1, alpha=0.7)
    ax.set_xlabel('Pwf (psi)', color='#CBD5E1', fontsize=9)
    ax.set_ylabel(ylabel, color='#CBD5E1', fontsize=9)
    ax.set_title(title, color='#CBD5E1', fontsize=10)
    ax.tick_params(colors='#64748B')
    for sp in ax.spines.values(): sp.set_color('#1E293B')
    ax.grid(color='#1E293B', ls='--', alpha=0.4)

ax3.axhline(10, color='#F59E0B', ls=':', lw=1, label='10% (warning)')
ax3.axhline(15, color='#EF4444', ls=':', lw=1, label='15% (gas lock)')
ax3.legend(facecolor='#111827', edgecolor='#1E293B', labelcolor='#CBD5E1', fontsize=8)

fig.suptitle('Sensibilidad del diseño BES a Pwf — decisión de yacimientos', 
             color='#CBD5E1', fontsize=11)
plt.tight_layout()
plt.show()

## Conclusiones del Módulo 9

### 1. Inversión pedagógica clave
- En M1–M8: Pwf es **resultado** del balance nodal IPR ∩ VLP
- En M9: Pwf es **decisión estratégica de yacimientos** → Q es consecuencia

### 2. Flujo secuencial con ciclos
El diseño BES no es lineal. Los CICLOS A–E permiten iterar sin perder el hilo:
- Si el GVF es alto → agregar separador (CICLO A) y recalcular Q_total
- Si el BEP ratio está fuera de 80–120% → cambiar frecuencia o serie (CICLO B)
- Si la holgura mecánica es insuficiente → CICLO E vuelve al PASO 4 con restricción de OD

### 3. Interrelación eléctrica-térmica-mecánica
El AWG del cable afecta OD → afecta holgura mecánica.  
La T° de fondo afecta V_drop (R_T) y vida útil de aislamiento (Arrhenius).  
El HP del motor determina la corriente → que determina el AWG mínimo.

### 4. IEEE 519-2014 desde el diseño
La elección del VSD en PASO 0 define si el sistema cumple THD < 5% desde el inicio.  
VSD de 18 pulsos (THD 4%) o AFE (THD 2.5%) son las únicas opciones que cumplen.